In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import glob
from scipy.stats import spearmanr
from statsmodels.stats.multitest import multipletests
import os

# TEST phenotypes

In [2]:
dpheno = pd.read_csv("../44_train_and_test/00_combine_samples_means.tsv", sep="\t", header=0)
dpheno = dpheno[dpheno['SET'] == 'TEST'].reset_index(drop=True)
print(dpheno.shape)
dpheno["sample"] = dpheno["POP_SITE"]
dpheno.head()

(2048, 8)


,Trait_name,POP_SITE,POP_ID,SITE_ID,SET,mean,sd,n,sample
0,Average_Ring_Density,1329_CH,1329,CH,TEST,476.575417,45.597986,8,1329_CH
1,Average_Ring_Density,1329_ML,1329,ML,TEST,480.010872,108.572664,3,1329_ML
2,Average_Ring_Density,1528_ML,1528,ML,TEST,514.502734,62.179284,3,1528_ML
3,Average_Ring_Density,1530_AC,1530,AC,TEST,553.321396,48.783433,14,1530_AC
4,Average_Ring_Density,1530_CH,1530,CH,TEST,526.616667,93.195567,6,1530_CH


# TRAIN phenotypes

In [3]:
tpheno = pd.read_csv("../44_train_and_test/00_combine_samples_means.tsv", sep="\t", header=0)
tpheno = tpheno[tpheno['SET'] == 'TRAIN'].reset_index(drop=True)
print(tpheno.shape)
tpheno["sample"] = tpheno["POP_SITE"]

print(tpheno[tpheno["n"] < 3].shape)

tpheno = tpheno[tpheno['n']>2].reset_index(drop=True)
print(tpheno.shape)

(1738, 8)
(108, 9)
(1630, 9)


# CTD

In [18]:
#dclim = pd.read_csv("../34_climate_transfer_distance/03_climate_distance_All_vars.tsv", sep="\t", header=0)
#dclim["sample"] = dclim.apply(lambda x: str(x["Provenance"]) + "_" + x["SITE_ID"], axis=1)
#dclim = dclim[["sample","EuclDist"]].reset_index(drop=True)
#dclim.head()

In [19]:
dclim = pd.read_csv("../34_climate_transfer_distance/03_climate_distance.tsv", sep="\t", header=0)
#dclim["sample"] = dclim.apply(lambda x: str(x["Provenance"]) + "_" + x["SITE_ID"], axis=1)
dclim = dclim.rename({"ECD_AC":"AC", "ECD_CH":"CH", "ECD_ML":"ML", "ECD_PR":"PR"}, axis=1)
gclim = dclim.set_index("POP_ID").stack().reset_index().rename(columns={"level_1":"SITE_ID",0:"EuclDist"})
gclim["sample"] = gclim.apply(lambda x: str(x["POP_ID"]) + "_" + x["SITE_ID"], axis=1)
gclim = gclim[["sample","EuclDist"]].reset_index(drop=True)
gclim.head()

,sample,EuclDist
0,1329_AC,6.201305
1,1329_CH,11.032084
2,1329_ML,6.242049
3,1329_PR,10.059107
4,1528_AC,2.694499


In [21]:
dm = pd.merge(dpheno, gclim, on = "sample", how = "left")
dm.head()

,Trait_name,POP_SITE,POP_ID,SITE_ID,SET,mean,sd,n,sample,EuclDist
0,Average_Ring_Density,1329_CH,1329,CH,TEST,476.575417,45.597986,8,1329_CH,11.032084
1,Average_Ring_Density,1329_ML,1329,ML,TEST,480.010872,108.572664,3,1329_ML,6.242049
2,Average_Ring_Density,1528_ML,1528,ML,TEST,514.502734,62.179284,3,1528_ML,4.039414
3,Average_Ring_Density,1530_AC,1530,AC,TEST,553.321396,48.783433,14,1530_AC,7.690994
4,Average_Ring_Density,1530_CH,1530,CH,TEST,526.616667,93.195567,6,1530_CH,11.128378


In [22]:
dm2 = pd.merge(tpheno, gclim, on = "sample", how = "left")
dm2.head()

,Trait_name,POP_SITE,POP_ID,SITE_ID,SET,mean,sd,n,sample,EuclDist
0,Average_Ring_Density,1329_CH,1329,CH,TRAIN,465.670982,65.589447,7,1329_CH,11.032084
1,Average_Ring_Density,1329_ML,1329,ML,TRAIN,513.398051,52.314507,13,1329_ML,6.242049
2,Average_Ring_Density,1528_ML,1528,ML,TRAIN,487.240829,65.902345,7,1528_ML,4.039414
3,Average_Ring_Density,1530_CH,1530,CH,TRAIN,518.358889,44.092654,6,1530_CH,11.128378
4,Average_Ring_Density,1530_ML,1530,ML,TRAIN,474.392630,65.059481,10,1530_ML,8.142808


# Reading genetic offsets

In [23]:
traits = ["Height", "Biomass_Increment", "Biomass_Increment_1980", "Biomass_Increment_1985", 
                "Biomass_Increment_1990", "Biomass_Increment_1995", "Biomass_Increment_2000", 
                "Biomass_Increment_2005", "Biomass_Increment_2010", "Biomass_Increment_2015",
                "Average_Ring_Density", "DBH", "Rs", "Rl", "Rr","Rc"]

gardens = ["PR","ML","CH","AC"]
dsets = ["1000"]

In [24]:
D = []

for trait in traits:
    for dset in dsets:
        for garden in gardens:
            file_path = "../48_garden_offset_run_separate/01_run_gradient_forest_" + garden + "_" + dset + "_" + trait + ".tsv"
            if os.path.exists(file_path):
                df_ = pd.read_csv(file_path, sep="\t", header=0)
                #df_["marker"] = dset
                df_["trait"] = trait
                df_["garden"] = garden
                D.append(df_)
dD = pd.concat(D)
print(dD.shape)
dD.head()

(2019, 16)


,Trait_name,sample,SITE_ID,group,PC1,PC2,PC3,now_at1,now_at2,now_at3,new_at1,new_at2,new_at3,offset,trait,garden
0,Height,2209_PR,PR,West,12.351204,-1.598833,0.851360,0.073840,0.013435,0.066650,0.035497,0.020165,0.006536,0.071618,Height,PR
1,Height,325_PR,PR,Central,2.506557,-3.166880,1.958590,0.044412,0.000946,0.093891,0.035497,0.020165,0.006536,0.089887,Height,PR
2,Height,4351_PR,PR,Central,-6.244846,1.531312,-1.399408,-0.004180,0.023815,0.008746,0.035497,0.020165,0.006536,0.039906,Height,PR
3,Height,4420_PR,PR,West,6.170760,3.960606,-2.183338,0.052893,0.032193,0.000328,0.035497,0.020165,0.006536,0.022041,Height,PR
4,Height,6805_PR,PR,East,0.111999,-2.575310,0.678848,0.027140,0.004848,0.063826,0.035497,0.020165,0.006536,0.059888,Height,PR


In [25]:
dD_sub = dD[["sample","SITE_ID","offset","trait","group"]].reset_index(drop=True)
dD_sub.head()

,sample,SITE_ID,offset,trait,group
0,2209_PR,PR,0.071618,Height,West
1,325_PR,PR,0.089887,Height,Central
2,4351_PR,PR,0.039906,Height,Central
3,4420_PR,PR,0.022041,Height,West
4,6805_PR,PR,0.059888,Height,East


### Test

In [26]:
dD_merge = pd.merge(dD_sub, dm, left_on = ["sample","SITE_ID","trait"], right_on = ["sample","SITE_ID","Trait_name"], how = "left")
print(dD_merge.shape)
dD_merge.head()

(2019, 13)


,sample,SITE_ID,offset,trait,group,Trait_name,POP_SITE,POP_ID,SET,mean,sd,n,EuclDist
0,2209_PR,PR,0.071618,Height,West,Height,2209_PR,2209,TEST,980.525258,302.158899,4,11.932738
1,325_PR,PR,0.089887,Height,Central,Height,325_PR,325,TEST,1239.891145,121.743290,8,6.805581
2,4351_PR,PR,0.039906,Height,Central,Height,4351_PR,4351,TEST,1423.598605,68.068593,4,7.864963
3,4420_PR,PR,0.022041,Height,West,Height,4420_PR,4420,TEST,678.153897,154.272486,7,5.398973
4,6805_PR,PR,0.059888,Height,East,Height,6805_PR,6805,TEST,1353.974749,229.855027,7,5.751414


### Train

In [27]:
tD_merge = pd.merge(dD_sub, dm2, left_on = ["sample","SITE_ID","trait"], right_on = ["sample","SITE_ID","Trait_name"], how = "left")
print(tD_merge.shape)
tD_merge.head()
tD_merge = tD_merge.dropna().reset_index(drop=True)
print(tD_merge.shape)

(2019, 13)
(1625, 13)


# Calculathing rho

In [28]:
def spearman_group(df, col_a, col_b, group_cols):
    results = []
    for name, g in df.groupby(group_cols):
        if len(g) < 3:
            rho, p = np.nan, np.nan
        elif ((len(g[col_a].dropna())<3) or (len(g[col_b].dropna())<3)):
            rho, p = np.nan, np.nan
        else:
            rho, p = spearmanr(g[col_a], g[col_b], nan_policy='omit')
        # name is tuple if multiple group columns
        name = name if isinstance(name, tuple) else (name,)
        results.append((*name, rho, p, len(g)))
            
    res_df = pd.DataFrame(results, columns=list(group_cols) + ['spearman_rho', 'p_value', 'n_samples'])
    
    # Adjust p-values
    mask = res_df['p_value'].notna()
    res_df['p_adj'] = np.nan
    if mask.any():
        res_df.loc[mask, 'p_adj'] = multipletests(res_df.loc[mask, 'p_value'], method='fdr_bh')[1]

    return res_df

# Test

In [29]:
df_cors = spearman_group(dD_merge, 'EuclDist', 'mean', ['SITE_ID','trait'])
df_cors['sign'] = df_cors['spearman_rho'].apply(lambda x: 'plus' if x >0 else 'minus')
df_cors["SET"] = "TEST"
df_cors.head()

,SITE_ID,trait,spearman_rho,p_value,n_samples,p_adj,sign,SET
0,AC,Average_Ring_Density,-0.278592,1.225870e-01,32,0.260417,minus,TEST
1,AC,Biomass_Increment,-0.657258,4.375210e-05,32,0.000394,minus,TEST
2,AC,Biomass_Increment_1980,-0.782662,1.405861e-06,27,0.000042,minus,TEST
3,AC,Biomass_Increment_1985,-0.784457,1.078083e-07,32,0.000007,minus,TEST
4,AC,Biomass_Increment_1990,-0.699047,8.561824e-06,32,0.000108,minus,TEST


# Train

In [30]:
dt_cors = spearman_group(tD_merge, 'EuclDist', 'mean', ['SITE_ID','trait'])
dt_cors['sign'] = dt_cors['spearman_rho'].apply(lambda x: 'plus' if x >0 else 'minus')
dt_cors["SET"] = "TRAIN"
dt_cors.head()

,SITE_ID,trait,spearman_rho,p_value,n_samples,p_adj,sign,SET
0,AC,Average_Ring_Density,-0.466667,0.205386,9,0.335104,minus,TRAIN
1,AC,Biomass_Increment,-0.750000,0.019942,9,0.067183,minus,TRAIN
2,AC,Biomass_Increment_1980,-0.500000,0.666667,3,0.794872,minus,TRAIN
3,AC,Biomass_Increment_1985,-0.566667,0.111633,9,0.204644,minus,TRAIN
4,AC,Biomass_Increment_1990,-0.583333,0.099186,9,0.198372,minus,TRAIN


In [31]:
dcombined = pd.concat([df_cors, dt_cors], ignore_index=True)
dcombined.head()

,SITE_ID,trait,spearman_rho,p_value,n_samples,p_adj,sign,SET
0,AC,Average_Ring_Density,-0.278592,1.225870e-01,32,0.260417,minus,TEST
1,AC,Biomass_Increment,-0.657258,4.375210e-05,32,0.000394,minus,TEST
2,AC,Biomass_Increment_1980,-0.782662,1.405861e-06,27,0.000042,minus,TEST
3,AC,Biomass_Increment_1985,-0.784457,1.078083e-07,32,0.000007,minus,TEST
4,AC,Biomass_Increment_1990,-0.699047,8.561824e-06,32,0.000108,minus,TEST


In [32]:
dcombined.to_csv("12_get_ctd.tsv", sep="\t", header=True, index=False)

# With cluster

### Test

In [33]:
df_cors = spearman_group(dD_merge, 'EuclDist', 'mean', ['SITE_ID','trait','group'])
df_cors['sign'] = df_cors['spearman_rho'].apply(lambda x: 'plus' if x >0 else 'minus')
df_cors["SET"] = "TEST"
df_cors.head()

,SITE_ID,trait,group,spearman_rho,p_value,n_samples,p_adj,sign,SET
0,AC,Average_Ring_Density,Central,-0.538462,0.070894,12,0.315574,minus,TEST
1,AC,Average_Ring_Density,East,-0.515152,0.127553,10,0.442530,minus,TEST
2,AC,Average_Ring_Density,ME,NaN,NaN,1,NaN,minus,TEST
3,AC,Average_Ring_Density,WI,NaN,NaN,1,NaN,minus,TEST
4,AC,Average_Ring_Density,West,-0.142857,0.787172,6,0.874636,minus,TEST


### Train

In [34]:
dt_cors = spearman_group(tD_merge, 'EuclDist', 'mean', ['SITE_ID','trait','group'])
dt_cors['sign'] = dt_cors['spearman_rho'].apply(lambda x: 'plus' if x >0 else 'minus')
dt_cors["SET"] = "TRAIN"
dt_cors.head()

,SITE_ID,trait,group,spearman_rho,p_value,n_samples,p_adj,sign,SET
0,AC,Average_Ring_Density,Central,-0.1,0.872889,5,0.912565,minus,TRAIN
1,AC,Average_Ring_Density,East,NaN,NaN,2,NaN,minus,TRAIN
2,AC,Average_Ring_Density,West,NaN,NaN,2,NaN,minus,TRAIN
3,AC,Biomass_Increment,Central,0.0,1.000000,5,1.000000,minus,TRAIN
4,AC,Biomass_Increment,East,NaN,NaN,2,NaN,minus,TRAIN


In [35]:
dcombined2 = pd.concat([df_cors, dt_cors], ignore_index=True)
dcombined2.head()

,SITE_ID,trait,group,spearman_rho,p_value,n_samples,p_adj,sign,SET
0,AC,Average_Ring_Density,Central,-0.538462,0.070894,12,0.315574,minus,TEST
1,AC,Average_Ring_Density,East,-0.515152,0.127553,10,0.442530,minus,TEST
2,AC,Average_Ring_Density,ME,NaN,NaN,1,NaN,minus,TEST
3,AC,Average_Ring_Density,WI,NaN,NaN,1,NaN,minus,TEST
4,AC,Average_Ring_Density,West,-0.142857,0.787172,6,0.874636,minus,TEST


In [36]:
dcombined2.to_csv("12_get_ctd_clusters.tsv", sep="\t", header=True, index=False)